# 1. Установка # Обновление пакета geomml

In [ ]:
from pathlib import Path
import os

root = Path(os.getcwd()).resolve().parents[1]  # вверх на 2 уровня
!python {str(root / "scripts" / "bootstrap.py")}

%load_ext autoreload
%autoreload 2

from hydra import initialize_config_dir, compose
from hydra.utils import instantiate

config_name = "default"

with initialize_config_dir(config_dir=str(root / "configs"), version_base="1.3"):
    cfg = compose(config_name=config_name)

In [ ]:
import os, time, copy, torch
import torch.nn.functional as F

from geomml.models import build as build_model
from geomml.datasets import build as build_dataset
from geomml.utils.loader import build as build_loaders
from geomml.trainers.evaluate_model import print_metrics
from geomml.utils.plot_history import *
from geomml.trainers.multi_task import MultiTaskTrainer
from geomml.losses.mae_mse import mae_mse

# Получение --- Скачивание датасета
dataset = build_dataset("homo")

# Разделение train/val/test # Подготовка DataLoader'ов
loaders = build_loaders(
    dataset, 
    batch_size=64,
    train_size=0.8,
    valid_size=0.1,
)

model = build_model("egnn_gap", num_atom_types=100, emb_dim=128, depth=4,)# hidden_dim=128,)
criterion = torch.nn.MSELoss()
 
''' Optimizer
* loss = MSE (для стабильного градиента) сильнее штрафует выбросы → лучше учит модель
* metric = MAE (для интерпретируемого качества) лучше отражает реальную ошибку в химических величинах
Использование метрики MAE вместе с MSE при валидации.
'''
optimizer = instantiate(cfg.optimizer, params=model.parameters())
 
''' Scheduler
Два варианта подходят для разных сценариев:
* **CosineAnnealingLR** (cosine LR scheduler) — заранее известное число эпох, скорость обучения плавно уменьшается по косинусу независимо от качества модели.
* **ReduceLROnPlateau** — уменьшает learning rate только тогда, когда валидационная ошибка перестает улучшаться. Для Early Stopping, этот вариант обычно предпочтительнее для регрессии на QM9.
'''
scheduler = instantiate(cfg.scheduler, optimizer=optimizer)

'''
loss_fn_dict={
    "mse": lambda pred, batch:
        F.mse_loss(pred["y"], batch["y"]),

    "dist": lambda pred, batch:
        dist_loss(pred, batch),

    "laplace": lambda pred, batch:
        laplace_loss(pred, batch),

    "contrastive": lambda pred, batch:
        contrastive_loss(pred, batch),
},

loss_fn_dict намного гибче: функции потерь могут использовать любые поля модели 
(pred["emb"], pred["coords"], pred["attention"]) и любые данные батча (batch.pos, batch.edge_index, batch.batch), 
без изменения интерфейса модели. 

Легко добавлять регуляризаторы вроде loss_dist, поскольку loss_fn_dict хорошо масштабируется и не требует «притворяться», что регуляризатор является обычной функцией потерь вида (prediction, target).
'''


trainer = MultiTaskTrainer(
    model=model,
    optimizer=optimizer,
    criterion=mae_mse(mae_w=0.8, mse_w=0.2),
    scheduler=scheduler,
    patience=15,        # сколько эпох ждать улучшения
    min_delta=1e-4,
    checkpoint_name="best-gap.pt",
)

# Обучение
model, history = trainer.fit(
    train_loader=loaders.train,
    valid_loader=loaders.valid,
    num_epochs=1000
)

plot_history(history, title="GAP model training mae_mse losses curves on HOMO-dataset")
print_metrics(model, loaders.test)

del optimizer, model, trainer, history, loaders, scheduler


Dataset size: 130831

Epoch [   1/1000] | TrainLoss: 0.3795 | ValidLoss: 0.2868 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 123.06s
Epoch [   2/1000] | TrainLoss: 0.2354 | ValidLoss: 0.1986 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 109.53s
Epoch [   3/1000] | TrainLoss: 0.2004 | ValidLoss: 0.1928 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 114.04s
Epoch [   4/1000] | TrainLoss: 0.1794 | ValidLoss: 0.1845 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 114.40s
Epoch [   5/1000] | TrainLoss: 0.1649 | ValidLoss: 0.1554 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 114.19s
Epoch [   6/1000] | TrainLoss: 0.1525 | ValidLoss: 0.1357 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 123.67s
Epoch [   7/1000] | TrainLoss: 0.1388 | ValidLoss: 0.1309 | LR: 1.00e-03 | EarlyStop: 0 /15 | EpochTime: 110.98s
Epoch [   8/1000] | TrainLoss: 0.1325 | ValidLoss: 0.1317 | LR: 1.00e-03 | EarlyStop: 1 /15 | EpochTime: 109.32s
Epoch [   9/1000] | TrainLoss: 0.1230 | ValidLoss: 0.1176 | LR: 1.00e-03 